In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Part 2 - Data Overview

In [ ]:
URI = "sqlite://cell-count-3nf.db"

# Load sample metadata and normalized long-format cell counts
samples_df = pl.read_database_uri(
    query="SELECT * FROM samples",
    uri=URI,
    engine="adbc"
)

cell_counts_df = pl.read_database_uri(
    query="SELECT sample, population, count FROM cell_counts",
    uri=URI,
    engine="adbc"
)

samples_df.head()


In [ ]:
cell_cols = ['b_cell', 'cd4_t_cell', 'cd8_t_cell', 'nk_cell', 'monocyte']

total_counts_df = cell_counts_df.group_by('sample').agg(pl.col('count').sum().alias('total_count'))

freq_df = (
    samples_df
    .join(total_counts_df, on='sample', how='inner')
    .join(cell_counts_df, on='sample', how='inner')
    .with_columns(
        percentage = (pl.col('count') / pl.col('total_count')) * 100
    )
)

freq_df.select(['sample', 'total_count', 'population', 'count', 'percentage']).head()


## Part 3 - Statistical Analysis

In [4]:
# Get the treatment_id for melanoma patients treated with miraclib
mela_mira_id = pl.read_database_uri(
    query="""
            SELECT treatment_id FROM treatments
            WHERE treatment = 'miraclib' and condition = 'melanoma'
            """,
    uri=URI,
    engine="adbc"
)['treatment_id'][0]

# Query the outcomes table and filter to only melanoma patients treated with miraclib
outcomes_filtered = pl.read_database_uri(
    query=f"""
            SELECT * FROM subject_outcomes 
            WHERE treatment_id = {mela_mira_id}
            """,
    uri=URI,
    engine="adbc"
)

outcomes_filtered.head()

subject,treatment_id,response
str,i64,str
"""sbj2811""",5,"""no"""
"""sbj2824""",5,"""no"""
"""sbj2635""",5,"""yes"""
"""sbj2181""",5,"""no"""
"""sbj364""",5,"""no"""


In [ ]:
# Join outcomes with the freq_df and filter to only PBMC samples
PBMC_merged_df = (
    freq_df
    .join(outcomes_filtered, left_on='subject', right_on='subject', how='inner')
    .filter(pl.col('sample_type') == 'PBMC')
    .drop('sample_type')
)

# Baseline-only is the primary response-prediction analysis to avoid post-treatment leakage.
baseline_PBMC_df = PBMC_merged_df.filter(pl.col('time_from_treatment_start') == 0)
baseline_PBMC_df.head()


In [ ]:
# Create the visualization for the baseline predictive cohort
g = sns.catplot(
    data=baseline_PBMC_df.to_pandas(),
    x="response", 
    y="percentage", 
    hue="response",
    col="population",    
    kind="box",           
    col_wrap=3,           
    palette="muted",
    height=4, 
    aspect=.8,
    sharey=False
)

# Clean up titles and labels
g.set_axis_labels("Response", "Frequency (%)")
g.set_titles("{col_name}")

plt.subplots_adjust(top=0.9)
g.fig.suptitle("Baseline PBMC Immune Cell Frequencies: Responders vs Non-Responders")

plt.show()


In [ ]:
def fdr_bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p), np.nan)
    valid = ~np.isnan(p)
    valid_p = p[valid]
    order = np.argsort(valid_p)
    ranks = np.arange(1, len(valid_p) + 1)
    ranked_adjusted = valid_p[order] * len(valid_p) / ranks
    ranked_adjusted = np.minimum.accumulate(ranked_adjusted[::-1])[::-1]
    valid_positions = np.where(valid)[0]
    adjusted[valid_positions[order]] = np.minimum(ranked_adjusted, 1.0)
    return adjusted

analysis_subject_df = (
    baseline_PBMC_df
    .group_by(['subject', 'response', 'population'])
    .agg(pl.col('percentage').mean().alias('percentage'))
)

# Run subject-level Welch t-test for each population
results = []

for cell in cell_cols:
    responders = analysis_subject_df.filter((pl.col('population') == cell) & (pl.col('response') == 'yes'))['percentage']
    non_responders = analysis_subject_df.filter((pl.col('population') == cell) & (pl.col('response') == 'no'))['percentage']
    
    t_stat, p_val = stats.ttest_ind(responders, non_responders, equal_var=False)
    _, mann_whitney_p = stats.mannwhitneyu(responders, non_responders, alternative='two-sided')
    
    mean_resp = responders.mean()
    mean_non_resp = non_responders.mean()
    
    results.append({
        'population': cell,
        'n_responders': responders.len(),
        'n_non_responders': non_responders.len(),
        'mean_responder_pct': round(mean_resp, 2),
        'mean_non_responder_pct': round(mean_non_resp, 2),
        'difference': round(mean_resp - mean_non_resp, 2),
        'p_value': p_val,
        'mann_whitney_p_value': mann_whitney_p
    })

results_df = pl.DataFrame(results).to_pandas()
results_df['fdr_q_value'] = fdr_bh_adjust(results_df['p_value'].to_numpy())
results_df['bonferroni_p_value'] = np.minimum(results_df['p_value'] * len(cell_cols), 1.0)
results_df = results_df.sort_values('fdr_q_value')


In [ ]:
results_df


## Part 4 - Data Subset Analysis

In [9]:
# samples filtered to only PBMC and time_from_treatment_start = 0
samps_filterd = samples_df.filter((pl.col('sample_type') == 'PBMC') & (pl.col('time_from_treatment_start') == 0))


# join the samples with the outcomes table that was previously filtered to only melanoma patients treated with miraclib
samps_outcomes_df = samps_filterd.join(outcomes_filtered, left_on='subject', right_on='subject', how='inner')

# join again on subjects table to get projects and sex
subjects_df = pl.read_database_uri(
    query="SELECT * FROM subjects",
    uri=URI,
    engine="adbc"
)


merged_samples_df = samps_outcomes_df.join(subjects_df, left_on='subject', right_on='subject', how='inner')[['sample', 'subject', 'project', 'response', 'sex']]
merged_samples_df.head()

sample,subject,project,response,sex
str,str,str,str,str
"""sample08133""","""sbj2711""","""prj3""","""no""","""M"""
"""sample08214""","""sbj2738""","""prj3""","""no""","""F"""
"""sample03720""","""sbj740""","""prj1""","""no""","""M"""
"""sample03414""","""sbj638""","""prj1""","""no""","""F"""
"""sample01311""","""sbj1306""","""prj1""","""yes""","""M"""


In [10]:
projects_df = pl.read_database_uri(
    query="SELECT project FROM projects ORDER BY project",
    uri=URI,
    engine="adbc"
)

proj_counts = (
    projects_df
    .join(
        merged_samples_df.group_by('project').agg(pl.col('sample').n_unique().alias('sample_count')),
        on='project',
        how='left'
    )
    .with_columns(pl.col('sample_count').fill_null(0).cast(pl.Int64))
)

response_counts = merged_samples_df.group_by("response").len()

sex_counts = merged_samples_df.group_by("sex").len() 

In [11]:
print(proj_counts, response_counts, sex_counts)

shape: (3, 2)
┌─────────┬──────────────┐
│ project ┆ sample_count │
│ ---     ┆ ---          │
│ str     ┆ i64          │
╞═════════╪══════════════╡
│ prj1    ┆ 384          │
│ prj2    ┆ 0            │
│ prj3    ┆ 272          │
└─────────┴──────────────┘ shape: (2, 2)
┌──────────┬─────┐
│ response ┆ len │
│ ---      ┆ --- │
│ str      ┆ u32 │
╞══════════╪═════╡
│ yes      ┆ 331 │
│ no       ┆ 325 │
└──────────┴─────┘ shape: (2, 2)
┌─────┬─────┐
│ sex ┆ len │
│ --- ┆ --- │
│ str ┆ u32 │
╞═════╪═════╡
│ M   ┆ 344 │
│ F   ┆ 312 │
└─────┴─────┘


## Part 5 - Average B Cells

In [ ]:
mela_b_cell_count = pl.read_database_uri(
    query="""
            SELECT ROUND(AVG(cc.count), 2) AS avg_b_cell
            FROM samples samp
            JOIN cell_counts cc ON samp.sample = cc.sample
            JOIN subjects sub ON samp.subject = sub.subject
            JOIN subject_outcomes out ON sub.subject = out.subject
            JOIN treatments t ON out.treatment_id = t.treatment_id
            WHERE t.condition = 'melanoma'
            AND cc.population = 'b_cell'
            AND samp.time_from_treatment_start = 0
            AND sub.sex = 'M'
            AND out.response = 'yes'
            """,
    uri=URI,
    engine="adbc"
)


In [13]:
mela_b_cell_count

avg_b_cell
f64
10206.15
